# 05. Optuna 재탐색 + 증강 조합 재실험 (파손 포함 데이터)

## 어떤 데이터로 돌리는가

실행 시점의 `data/processed`를 그대로 사용합니다.
이번 계획에서는 **clean + damage 를 모두 담아 데이터 수가 약 2배인 데이터셋**입니다.

아래 "데이터 확인" 셀에서 train 장수를 꼭 보고, 의도한 데이터가 맞는지 확인하세요.
(clean 전용은 약 7,600장 / 2배 데이터는 약 15,500장)

## 왜 다시 튜닝하는가

01번 Optuna는 **86개 카테고리** 데이터로 탐색한 결과입니다.
그 값을 17개 카테고리에 그대로 적용한 결과가 지금까지 계속 손해였습니다.

```text
17개 clean        B00 증강OFF+auto 0.8265  vs  B03 증강OFF+tuned 0.7858   (-0.0407)
17개 파손포함 2배   B00 증강OFF+auto 0.8463  vs  B03 증강OFF+tuned 0.8336   (-0.0127)
```

즉 **86개 기준으로 찾은 하이퍼파라미터는 지금 데이터에서 유효하지 않습니다.**
현재 데이터로 다시 탐색할 근거가 충분합니다.

## 01번 결과에서 가져온 교훈 (trial 수를 줄이는 근거)

01번의 COMPLETE trial 19개를 보면 패턴이 뚜렷합니다.

| 관찰 | 내용 |
|---|---|
| optimizer별 적정 lr 대역이 완전히 다름 | AdamW에 SGD용 `lr0=0.008`을 준 trial은 mAP **0.0675**로 붕괴 |
| 극단적으로 낮은 lr0는 무조건 실패 | `lr0=1e-5` trial은 mAP **0.0048** |
| AdamW는 낮은 lr(0.0004~0.0011)만 시도되어 불리하게 평가됨 | AdamW 5개 평균 0.342 vs SGD 14개 평균 0.445 |
| 중요도 1·2위는 `lrf`, `lr0` | `optimizer`, `cos_lr` 자체의 중요도는 낮음 |

그래서 이번에는 **optimizer에 따라 `lr0` 탐색 범위를 나눕니다.**
"AdamW + 큰 lr" 같은 확실한 실패 조합에 trial을 낭비하지 않으므로
같은 25 trial 로 01번보다 훨씬 촘촘하게 탐색됩니다.

추가로 좋은 출발점 두 개를 미리 넣어둡니다(`enqueue_trial`).

1. `auto`가 실제로 고르는 설정과 비슷한 AdamW 조합
2. 01번에서 1위였던 SGD 조합

## 실험 순서

```text
1) Optuna 25 trial          (10 epoch, 증강 전부 OFF)
        ↓
2) 상위 3개 x seed 3개 재학습 (10 epoch)
        ↓
3) 평균 1위만 15 epoch 1회
        ↓
4) 새 tuned로 재실험 (대조군 B00~B03 4개 + 증강 33개 = 37개, 10 epoch)
        ↓
5) 하이퍼파라미터 top2 + 증강 top2 저장  -> 06번이 교차 실험에 사용
```

출력 폴더는 `../models/yolo/05_retune_optuna_augmentation/` 입니다.


## 1. 라이브러리

In [ ]:
from __future__ import annotations

import json
import math
import time
import random
import hashlib
import shutil
import gc
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import yaml
import torch

from ultralytics import YOLO

try:
    import ultralytics
    ULTRALYTICS_VERSION = ultralytics.__version__
except Exception:
    ULTRALYTICS_VERSION = "unknown"

try:
    from ultralytics.cfg import DEFAULT_CFG_DICT
except Exception:
    DEFAULT_CFG_DICT = {}

from IPython.display import display, Image as IPImage

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 500)


def set_korean_font():
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
    installed = {font.name for font in fm.fontManager.ttflist}

    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break

    plt.rcParams["axes.unicode_minus"] = False


set_korean_font()

print("Ultralytics:", ULTRALYTICS_VERSION)
print("PyTorch    :", torch.__version__)
print("OpenCV     :", cv2.__version__)

In [ ]:
import optuna
from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Optuna:", optuna.__version__)


## 2. 경로와 공통 설정

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# ------------------------------------------------------------
# 1) 1번 노트북이 만든 processed 데이터
# ------------------------------------------------------------
PROCESSED_DIR = (
    PROJECT_ROOT / "../../data/processed"
).resolve()

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"processed 폴더가 없습니다: {PROCESSED_DIR}\n"
        "먼저 01_recycling_eda_preprocess_build_processed.ipynb를 실행하세요."
    )

DATA_YAML = PROCESSED_DIR / "data.yaml"

# ------------------------------------------------------------
# 2) 모델/실험 산출물 위치
# ------------------------------------------------------------
EXPERIMENT_ROOT = (
    PROJECT_ROOT / "../models/yolo/05_retune_optuna_augmentation"
).resolve()

RUNS_DIR = EXPERIMENT_ROOT / "runs"
CV_CACHE_DIR = EXPERIMENT_ROOT / "opencv_datasets"

# ------------------------------------------------------------
# 3) 사람이 확인할 보고서 위치
# ------------------------------------------------------------
REPORT_ROOT = EXPERIMENT_ROOT / "report"
REPORT_SOURCE_DIR = REPORT_ROOT / "preprocess"
SUMMARY_DIR = REPORT_ROOT / "summary"
PER_CLASS_DIR = REPORT_ROOT / "per_class"
FINAL_DIR = REPORT_ROOT / "final_best"

for path in [
    EXPERIMENT_ROOT,
    RUNS_DIR,
    CV_CACHE_DIR,
    REPORT_ROOT,
    REPORT_SOURCE_DIR,
    SUMMARY_DIR,
    PER_CLASS_DIR,
    FINAL_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "yolo26n.pt"
EPOCHS = 10
IMGSZ = 640
BATCH = 8
PATIENCE = 12

# Optuna 최종 1위 하이퍼파라미터를 마지막에 한 번 더 돌릴 때 쓰는 epoch 수입니다.
# 02번의 15 epoch 결과와 같은 조건으로 비교하기 위해 15로 맞춥니다.
FINAL_COMPARE_EPOCHS = 15

# Optuna trial 1회당 epoch
TUNE_EPOCHS = 10
TUNE_PATIENCE = TUNE_EPOCHS

# 01번과 같은 25 trial 을 돌립니다. optimizer별로 lr 대역을 나눠 확실한 실패
# 조합에는 trial 을 낭비하지 않으므로, 같은 25 trial 로 01번보다 촘촘히 탐색됩니다.
TARGET_TOTAL_TRIALS = 25
TOP_K_FOR_MULTI_SEED = 3
MULTI_SEEDS = [42, 123, 777]

# trial 체크포인트는 평가 후 삭제해 디스크를 아낍니다.
KEEP_TUNE_RUNS = False

# 증강 후보를 얼마나 넓게 볼지 정합니다.
#   True  : 02번 카탈로그의 증강 33개를 전부 재실험 (epoch 10 이라 감당 가능)
#   False : 02번 순위 상위 AUG_SHORTLIST_TOP_N 개만
USE_ALL_AUGMENTATIONS = True
AUG_SHORTLIST_TOP_N = 8

# close_mosaic 은 '마지막 N epoch 동안 mosaic/mixup/cutmix 를 끈다'는 뜻입니다.
# YOLO_AUG_CONFIGS 에 적어둔 값과 같게 맞춰, aug=None 인 실험(B01/B02)에도
# 같은 조건이 적용되도록 합니다.
CLOSE_MOSAIC_EPOCHS = 3
# 데이터 로딩을 별도 프로세스로 병렬화합니다 (물리 14코어).
# 46개 런 전부 같은 값으로 돌려야 실험 간 비교가 공정합니다.
WORKERS = 4
SEED = 42

RUN_EXPERIMENTS = True
RUN_MULTI_SEED = False
SKIP_COMPLETED = True
SMOKE_TEST = False

# OpenCV static augmentation은 원본과 같은 데이터 개수를 유지합니다.
CV_APPLY_PROBABILITY = 0.80
CV_OUTPUT_JPEG_QUALITY = 95
CLEANUP_CV_DATASET_AFTER_RUN = True

if SMOKE_TEST:
    EPOCHS = 2
    PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("PROCESSED_DIR  :", PROCESSED_DIR)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("RUNS_DIR       :", RUNS_DIR)
print("REPORT_ROOT    :", REPORT_ROOT)
print("SUMMARY_DIR    :", SUMMARY_DIR)
print("MODEL_NAME     :", MODEL_NAME)
print("EPOCHS         :", EPOCHS)
print("FINAL_COMPARE  :", FINAL_COMPARE_EPOCHS)
print("IMGSZ          :", IMGSZ)
print("BATCH          :", BATCH)
print("SMOKE_TEST     :", SMOKE_TEST)

TUNE_RUNS_DIR = EXPERIMENT_ROOT / "tune_runs"
OPTUNA_REPORT_DIR = REPORT_ROOT / "optuna"
OPTUNA_FIG_DIR = OPTUNA_REPORT_DIR / "figures"

for path in [TUNE_RUNS_DIR, OPTUNA_REPORT_DIR, OPTUNA_FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

STUDY_DB = EXPERIMENT_ROOT / "optuna_study_17class.db"

print("TUNE_EPOCHS    :", TUNE_EPOCHS)
print("TARGET_TRIALS  :", TARGET_TOTAL_TRIALS)
print("STUDY_DB       :", STUDY_DB)


## 3. GPU 확인

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {total_vram:.2f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS 사용")
else:
    DEVICE = "cpu"
    print("CPU 사용")

print("DEVICE =", DEVICE)

## 4. data.yaml 보정

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

dataset_config["path"] = str(PROCESSED_DIR.resolve())

RUNTIME_DATA_YAML = SUMMARY_DIR / "runtime_processed.yaml"

with open(RUNTIME_DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, allow_unicode=True, sort_keys=False)

names_raw = dataset_config["names"]

if isinstance(names_raw, dict):
    CLASS_NAMES = {int(key): str(value) for key, value in names_raw.items()}
else:
    CLASS_NAMES = {index: str(value) for index, value in enumerate(names_raw)}

NUM_CLASSES = len(CLASS_NAMES)

print(RUNTIME_DATA_YAML.read_text(encoding="utf-8")[:5000])
print("클래스 수:", NUM_CLASSES)

## 5. 품질 보고서 (있으면)

In [ ]:
CLASS_SUPPORT_CSV = REPORT_SOURCE_DIR / "class_support_processed.csv"
PREPROCESS_SUMMARY_CSV = REPORT_SOURCE_DIR / "preprocess_summary.csv"

if CLASS_SUPPORT_CSV.exists():
    class_support_df = pd.read_csv(CLASS_SUPPORT_CSV)
    display(class_support_df)

    no_val_classes = class_support_df[class_support_df["val"].eq(0)]
    print("Validation object가 0인 클래스:", len(no_val_classes))
    if len(no_val_classes):
        display(no_val_classes[["class_name", "train", "val"]])
else:
    class_support_df = pd.DataFrame()
    print("class_support_processed.csv를 찾지 못했습니다.")

if PREPROCESS_SUMMARY_CSV.exists():
    display(pd.read_csv(PREPROCESS_SUMMARY_CSV))

## 6. processed 데이터 확인

In [ ]:
TRAIN_IMAGE_DIR = PROCESSED_DIR / "images" / "train"
VAL_IMAGE_DIR = PROCESSED_DIR / "images" / "val"
TRAIN_LABEL_DIR = PROCESSED_DIR / "labels" / "train"
VAL_LABEL_DIR = PROCESSED_DIR / "labels" / "val"

for path in [TRAIN_IMAGE_DIR, VAL_IMAGE_DIR, TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)

train_images = sorted(path for path in TRAIN_IMAGE_DIR.iterdir() if path.is_file())
val_images = sorted(path for path in VAL_IMAGE_DIR.iterdir() if path.is_file())
train_labels = sorted(TRAIN_LABEL_DIR.glob("*.txt"))
val_labels = sorted(VAL_LABEL_DIR.glob("*.txt"))

print("Train images:", len(train_images))
print("Train labels:", len(train_labels))
print("Val images  :", len(val_images))
print("Val labels  :", len(val_labels))

if len(train_images) != len(train_labels):
    raise ValueError("Train image와 label 개수가 다릅니다.")

if len(val_images) != len(val_labels):
    raise ValueError("Validation image와 label 개수가 다릅니다.")

if len(val_images) == 0:
    raise ValueError("Validation 이미지가 없습니다.")

print("Processed quick check: PASSED")

## 7. 증강 설정 정의

In [ ]:
AUGMENTATION_KEYS = [
    "hsv_h", "hsv_s", "hsv_v",
    "degrees", "translate", "scale", "shear", "perspective",
    "flipud", "fliplr", "bgr",
    "mosaic", "mixup", "cutmix", "copy_paste",
    "close_mosaic", "augmentations",
]

installed_aug_defaults = {
    key: DEFAULT_CFG_DICT.get(key, "<not available>")
    for key in AUGMENTATION_KEYS
}

display(
    pd.DataFrame({
        "argument": list(installed_aug_defaults.keys()),
        "installed_default": list(installed_aug_defaults.values()),
    })
)

In [ ]:
NO_AUG = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "bgr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 0,
    "augmentations": [],
}


def with_no_aug(**changes):
    config = dict(NO_AUG)
    config.update(changes)
    return config


def supported_train_args(config: dict | None):
    """현재 Ultralytics 버전에서 지원되는 key만 남깁니다."""
    if config is None:
        return {}, []

    if not DEFAULT_CFG_DICT:
        # config dictionary를 가져오지 못한 버전에서는 그대로 전달합니다.
        return dict(config), []

    supported = set(DEFAULT_CFG_DICT.keys())
    unknown = sorted(set(config.keys()) - supported)
    filtered = {key: value for key, value in config.items() if key in supported}

    return filtered, unknown

In [ ]:
# NOTE: close_mosaic 은 '마지막 N 에폭 동안 mosaic/mixup/cutmix 를 끈다'는 뜻이다.
# EPOCHS 와 같은 값이면 첫 에폭부터 꺼져 이 설정들이 전부 무증강으로 붕괴한다.
# (02번에서 Y08/Y09/Y10/Y14 지표가 완전히 같았던 원인) 그래서 3 으로 둔다.
YOLO_AUG_CONFIGS = {
    "hsv": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.50,
        hsv_v=0.35,
    ),
    "flip": with_no_aug(
        fliplr=0.50,
    ),
    "rotation": with_no_aug(
        degrees=10.0,
    ),
    "translate": with_no_aug(
        translate=0.08,
    ),
    "scale": with_no_aug(
        scale=0.25,
    ),
    "shear": with_no_aug(
        shear=2.0,
    ),
    "perspective": with_no_aug(
        perspective=0.0005,
    ),
    "mosaic": with_no_aug(
        mosaic=0.70,
        close_mosaic=3,
    ),
    "mixup": with_no_aug(
        mixup=0.15,
        close_mosaic=3,
    ),
    "cutmix": with_no_aug(
        cutmix=0.15,
        close_mosaic=3,
    ),
    "hsv_flip": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.50,
        hsv_v=0.35,
        fliplr=0.50,
    ),
    "geo_combo": with_no_aug(
        degrees=10.0,
        translate=0.08,
        scale=0.25,
        shear=2.0,
        perspective=0.0005,
        fliplr=0.50,
    ),
    "mosaic_hsv_geo": with_no_aug(
        hsv_h=0.015,
        hsv_s=0.45,
        hsv_v=0.30,
        degrees=8.0,
        translate=0.06,
        scale=0.20,
        fliplr=0.50,
        mosaic=0.65,
        close_mosaic=3,
    ),
    "mosaic_mixup_cutmix": with_no_aug(
        mosaic=0.65,
        mixup=0.10,
        cutmix=0.10,
        close_mosaic=3,
    ),
    "balanced_combo": with_no_aug(
        hsv_h=0.012,
        hsv_s=0.40,
        hsv_v=0.30,
        degrees=7.0,
        translate=0.06,
        scale=0.18,
        shear=1.0,
        perspective=0.0003,
        fliplr=0.45,
        mosaic=0.50,
        mixup=0.05,
        cutmix=0.05,
        close_mosaic=3,
    ),
}

## 8. OpenCV 증강 함수와 static dataset 생성기

02번 상위 후보에 OpenCV 계열(C/H)이 포함될 수 있으므로 그대로 가져옵니다.

In [ ]:
def image_to_label_path(image_path: Path) -> Path:
    parts = list(image_path.parts)
    indices = [index for index, part in enumerate(parts) if part.lower() == "images"]

    if not indices:
        raise ValueError(f"images 폴더가 경로에 없습니다: {image_path}")

    parts[indices[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


def read_yolo_label(label_path: Path, image_width: int, image_height: int):
    classes = []
    boxes = []

    text = label_path.read_text(encoding="utf-8").strip()

    for line in text.splitlines():
        class_id, xc, yc, bw, bh = map(float, line.split())
        class_id = int(class_id)

        x1 = (xc - bw / 2) * image_width
        y1 = (yc - bh / 2) * image_height
        x2 = (xc + bw / 2) * image_width
        y2 = (yc + bh / 2) * image_height

        classes.append(class_id)
        boxes.append([x1, y1, x2, y2])

    return (
        np.asarray(classes, dtype=int),
        np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
    )


def boxes_to_yolo_lines(classes, boxes, image_width: int, image_height: int):
    lines = []

    for class_id, box in zip(classes, boxes):
        x1, y1, x2, y2 = map(float, box)

        xc = ((x1 + x2) / 2) / image_width
        yc = ((y1 + y2) / 2) / image_height
        bw = (x2 - x1) / image_width
        bh = (y2 - y1) / image_height

        lines.append(
            f"{int(class_id)} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}"
        )

    return lines

In [ ]:
def bbox_area(boxes: np.ndarray):
    if len(boxes) == 0:
        return np.empty((0,), dtype=np.float32)

    widths = np.clip(boxes[:, 2] - boxes[:, 0], 0, None)
    heights = np.clip(boxes[:, 3] - boxes[:, 1], 0, None)
    return widths * heights


def transform_boxes(
    boxes: np.ndarray,
    matrix: np.ndarray,
    image_width: int,
    image_height: int,
    min_visible: float = 0.25,
    min_size: float = 2.0,
):
    if len(boxes) == 0:
        return boxes.copy(), np.empty((0,), dtype=bool)

    corners = np.stack(
        [
            boxes[:, [0, 1]],
            boxes[:, [2, 1]],
            boxes[:, [2, 3]],
            boxes[:, [0, 3]],
        ],
        axis=1,
    ).astype(np.float32)

    points = corners.reshape(-1, 1, 2)

    if matrix.shape == (2, 3):
        transformed = cv2.transform(points, matrix).reshape(-1, 4, 2)
    else:
        transformed = cv2.perspectiveTransform(points, matrix).reshape(-1, 4, 2)

    raw_boxes = np.column_stack([
        transformed[:, :, 0].min(axis=1),
        transformed[:, :, 1].min(axis=1),
        transformed[:, :, 0].max(axis=1),
        transformed[:, :, 1].max(axis=1),
    ]).astype(np.float32)

    raw_area = bbox_area(raw_boxes)

    clipped = raw_boxes.copy()
    clipped[:, [0, 2]] = np.clip(clipped[:, [0, 2]], 0, image_width)
    clipped[:, [1, 3]] = np.clip(clipped[:, [1, 3]], 0, image_height)

    clipped_area = bbox_area(clipped)
    visible_ratio = clipped_area / np.maximum(raw_area, 1e-6)

    keep = (
        ((clipped[:, 2] - clipped[:, 0]) >= min_size)
        & ((clipped[:, 3] - clipped[:, 1]) >= min_size)
        & (visible_ratio >= min_visible)
    )

    return clipped[keep], keep

In [ ]:
def cv_brightness_contrast(image, boxes, classes, rng):
    alpha = float(rng.uniform(0.75, 1.25))
    beta = float(rng.uniform(-30, 30))

    output = np.clip(
        image.astype(np.float32) * alpha + beta,
        0,
        255,
    ).astype(np.uint8)

    return output, boxes.copy(), classes.copy()


def cv_gamma(image, boxes, classes, rng):
    gamma = float(rng.uniform(0.70, 1.40))
    inverse_gamma = 1.0 / gamma

    table = np.array([
        ((value / 255.0) ** inverse_gamma) * 255
        for value in np.arange(256)
    ]).astype(np.uint8)

    output = cv2.LUT(image, table)
    return output, boxes.copy(), classes.copy()


def cv_clahe(image, boxes, classes, rng):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=float(rng.uniform(1.5, 3.0)),
        tileGridSize=(8, 8),
    )

    enhanced_l = clahe.apply(l_channel)
    output = cv2.cvtColor(
        cv2.merge([enhanced_l, a_channel, b_channel]),
        cv2.COLOR_LAB2BGR,
    )

    return output, boxes.copy(), classes.copy()


def cv_gaussian_blur(image, boxes, classes, rng):
    kernel_size = int(rng.choice([3, 5]))
    output = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return output, boxes.copy(), classes.copy()


def cv_motion_blur(image, boxes, classes, rng):
    kernel_size = int(rng.choice([5, 7]))
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    direction = int(rng.integers(0, 4))

    if direction == 0:
        kernel[kernel_size // 2, :] = 1.0
    elif direction == 1:
        kernel[:, kernel_size // 2] = 1.0
    elif direction == 2:
        np.fill_diagonal(kernel, 1.0)
    else:
        np.fill_diagonal(np.fliplr(kernel), 1.0)

    kernel /= kernel.sum()
    output = cv2.filter2D(image, -1, kernel)
    return output, boxes.copy(), classes.copy()


def cv_gaussian_noise(image, boxes, classes, rng):
    sigma = float(rng.uniform(5.0, 18.0))
    noise = rng.normal(0, sigma, size=image.shape).astype(np.float32)

    output = np.clip(
        image.astype(np.float32) + noise,
        0,
        255,
    ).astype(np.uint8)

    return output, boxes.copy(), classes.copy()


def cv_jpeg_compression(image, boxes, classes, rng):
    quality = int(rng.integers(45, 86))
    success, encoded = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, quality],
    )

    if not success:
        return image.copy(), boxes.copy(), classes.copy()

    output = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    return output, boxes.copy(), classes.copy()

In [ ]:
def cv_affine(image, boxes, classes, rng):
    height, width = image.shape[:2]

    angle = float(rng.uniform(-12, 12))
    scale = float(rng.uniform(0.88, 1.12))
    translate_x = float(rng.uniform(-0.07, 0.07) * width)
    translate_y = float(rng.uniform(-0.07, 0.07) * height)

    matrix = cv2.getRotationMatrix2D(
        (width / 2, height / 2),
        angle,
        scale,
    )
    matrix[0, 2] += translate_x
    matrix[1, 2] += translate_y

    output = cv2.warpAffine(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    new_boxes, keep = transform_boxes(
        boxes,
        matrix,
        width,
        height,
    )

    return output, new_boxes, classes[keep]


def cv_perspective(image, boxes, classes, rng):
    height, width = image.shape[:2]
    jitter = 0.035

    source = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ], dtype=np.float32)

    destination = source.copy()
    destination[:, 0] += rng.uniform(-jitter * width, jitter * width, 4)
    destination[:, 1] += rng.uniform(-jitter * height, jitter * height, 4)

    matrix = cv2.getPerspectiveTransform(
        source,
        destination.astype(np.float32),
    )

    output = cv2.warpPerspective(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    new_boxes, keep = transform_boxes(
        boxes,
        matrix,
        width,
        height,
    )

    return output, new_boxes, classes[keep]


def cv_horizontal_flip(image, boxes, classes, rng):
    height, width = image.shape[:2]
    output = cv2.flip(image, 1)
    new_boxes = boxes.copy()

    if len(new_boxes):
        old_x1 = boxes[:, 0].copy()
        old_x2 = boxes[:, 2].copy()
        new_boxes[:, 0] = width - old_x2
        new_boxes[:, 2] = width - old_x1

    return output, new_boxes, classes.copy()

In [ ]:
def cv_reencode_control(image, boxes, classes, rng):
    """픽셀 변환 없이 OpenCV decode/re-encode 영향만 측정하는 control입니다."""
    return image.copy(), boxes.copy(), classes.copy()


def cv_photometric_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = image.copy(), boxes.copy(), classes.copy()

    if rng.random() < 0.80:
        output, new_boxes, new_classes = cv_brightness_contrast(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.35:
        output, new_boxes, new_classes = cv_gamma(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.30:
        output, new_boxes, new_classes = cv_clahe(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.25:
        output, new_boxes, new_classes = cv_gaussian_noise(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.20:
        output, new_boxes, new_classes = cv_jpeg_compression(
            output, new_boxes, new_classes, rng
        )

    return output, new_boxes, new_classes


def cv_geometric_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = image.copy(), boxes.copy(), classes.copy()

    if rng.random() < 0.80:
        output, new_boxes, new_classes = cv_affine(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.30:
        output, new_boxes, new_classes = cv_perspective(
            output, new_boxes, new_classes, rng
        )

    if rng.random() < 0.45:
        output, new_boxes, new_classes = cv_horizontal_flip(
            output, new_boxes, new_classes, rng
        )

    return output, new_boxes, new_classes


def cv_mixed_combo(image, boxes, classes, rng):
    output, new_boxes, new_classes = cv_photometric_combo(
        image, boxes, classes, rng
    )
    output, new_boxes, new_classes = cv_geometric_combo(
        output, new_boxes, new_classes, rng
    )
    return output, new_boxes, new_classes


CV_POLICIES = {
    "reencode_control": cv_reencode_control,
    "brightness_contrast": cv_brightness_contrast,
    "gamma": cv_gamma,
    "clahe": cv_clahe,
    "gaussian_blur": cv_gaussian_blur,
    "motion_blur": cv_motion_blur,
    "gaussian_noise": cv_gaussian_noise,
    "jpeg_compression": cv_jpeg_compression,
    "affine": cv_affine,
    "perspective": cv_perspective,
    "horizontal_flip": cv_horizontal_flip,
    "photometric_combo": cv_photometric_combo,
    "geometric_combo": cv_geometric_combo,
    "mixed_combo": cv_mixed_combo,
}

In [ ]:
def stable_seed(*parts) -> int:
    text = "|".join(map(str, parts))
    return int(hashlib.sha1(text.encode("utf-8")).hexdigest()[:8], 16)


def build_opencv_dataset(
    policy_name: str,
    seed: int,
    apply_probability: float = CV_APPLY_PROBABILITY,
    overwrite: bool = False,
):
    if policy_name not in CV_POLICIES:
        raise KeyError(policy_name)

    policy_fn = CV_POLICIES[policy_name]
    dataset_dir = CV_CACHE_DIR / policy_name / f"seed_{seed}"
    image_dir = dataset_dir / "images" / "train"
    label_dir = dataset_dir / "labels" / "train"
    report_path = dataset_dir / "generation_report.csv"
    yaml_path = dataset_dir / "data.yaml"

    if yaml_path.exists() and report_path.exists() and not overwrite:
        return yaml_path

    if dataset_dir.exists() and overwrite:
        shutil.rmtree(dataset_dir)

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for source_image_path in train_images:
        source_label_path = image_to_label_path(source_image_path)
        image = cv2.imread(str(source_image_path))

        if image is None:
            raise ValueError(f"이미지를 읽지 못했습니다: {source_image_path}")

        height, width = image.shape[:2]
        classes, boxes = read_yolo_label(source_label_path, width, height)

        image_seed = stable_seed(seed, policy_name, source_image_path.name)
        rng = np.random.default_rng(image_seed)
        apply_aug = rng.random() < apply_probability

        if apply_aug:
            augmented_image, augmented_boxes, augmented_classes = policy_fn(
                image, boxes, classes, rng
            )

            # 기하 변환으로 원래 있던 모든 객체가 사라지면 너무 공격적인 샘플이므로 원본으로 fallback합니다.
            if len(classes) > 0 and len(augmented_classes) == 0:
                apply_aug = False
            else:
                output_image_path = image_dir / f"{source_image_path.stem}.jpg"
                success = cv2.imwrite(
                    str(output_image_path),
                    augmented_image,
                    [cv2.IMWRITE_JPEG_QUALITY, CV_OUTPUT_JPEG_QUALITY],
                )

                if not success:
                    raise IOError(f"이미지 저장 실패: {output_image_path}")

                output_label_path = label_dir / f"{source_image_path.stem}.txt"
                output_label_path.write_text(
                    "\n".join(
                        boxes_to_yolo_lines(
                            augmented_classes,
                            augmented_boxes,
                            width,
                            height,
                        )
                    ),
                    encoding="utf-8",
                )

                rows.append({
                    "source_image": source_image_path.name,
                    "output_image": output_image_path.name,
                    "augmented": True,
                    "source_objects": len(classes),
                    "output_objects": len(augmented_classes),
                })

        if not apply_aug:
            output_image_path = image_dir / source_image_path.name
            shutil.copy2(source_image_path, output_image_path)

            output_label_path = label_dir / source_label_path.name
            shutil.copy2(source_label_path, output_label_path)

            rows.append({
                "source_image": source_image_path.name,
                "output_image": output_image_path.name,
                "augmented": False,
                "source_objects": len(classes),
                "output_objects": len(classes),
            })

    generation_df = pd.DataFrame(rows)
    generation_df.to_csv(report_path, index=False, encoding="utf-8-sig")

    # train은 OpenCV static dataset, val은 원본 processed val을 그대로 사용합니다.
    cv_dataset_yaml = {
        "train": str(image_dir.resolve()),
        "val": str(VAL_IMAGE_DIR.resolve()),
        "names": {class_id: name for class_id, name in CLASS_NAMES.items()},
    }

    with open(yaml_path, "w", encoding="utf-8") as file:
        yaml.safe_dump(cv_dataset_yaml, file, allow_unicode=True, sort_keys=False)

    if len(generation_df) != len(train_images):
        raise ValueError("OpenCV dataset의 이미지 수가 원본 train과 다릅니다.")

    return yaml_path

## 9. 02번 실험 카탈로그

증강 조합의 설정을 찾아오기 위한 전체 목록입니다.

In [ ]:
EXPERIMENTS = [
    # Controls
    {
        "id": "B00", "name": "no_aug_auto", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": NO_AUG, "required_args": [],
        "hp_mode": "auto",
        "description": "00번 baseline 재현: 증강 전부 OFF + optimizer auto",
    },
    {
        "id": "B01", "name": "yolo_default_aug_auto", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": None, "required_args": [],
        "hp_mode": "auto",
        "description": "YOLO 기본 augmentation + optimizer auto",
    },
    {
        "id": "B02", "name": "yolo_default_aug_tuned", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": None, "required_args": [],
        "hp_mode": "tuned",
        "description": "YOLO 기본 augmentation + Optuna tuned 하이퍼파라미터",
    },
    {
        "id": "B03", "name": "no_aug_tuned", "family": "baseline",
        "data_kind": "processed", "cv_policy": None,
        "aug": NO_AUG, "required_args": [],
        "hp_mode": "tuned",
        "description": "증강 전부 OFF + Optuna tuned (순수 하이퍼파라미터 효과)",
    },

    # YOLO single-factor screening
    {
        "id": "Y01", "name": "yolo_hsv", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["hsv"], "required_args": ["hsv_h", "hsv_s", "hsv_v"],
        "description": "HSV 색조/채도/밝기 변화",
    },
    {
        "id": "Y02", "name": "yolo_horizontal_flip", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["flip"], "required_args": ["fliplr"],
        "description": "좌우 반전",
    },
    {
        "id": "Y03", "name": "yolo_rotation", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["rotation"], "required_args": ["degrees"],
        "description": "약한 회전",
    },
    {
        "id": "Y04", "name": "yolo_translate", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["translate"], "required_args": ["translate"],
        "description": "객체 위치 이동",
    },
    {
        "id": "Y05", "name": "yolo_scale", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["scale"], "required_args": ["scale"],
        "description": "확대/축소",
    },
    {
        "id": "Y06", "name": "yolo_shear", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["shear"], "required_args": ["shear"],
        "description": "약한 shear 기울임",
    },
    {
        "id": "Y07", "name": "yolo_perspective", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["perspective"], "required_args": ["perspective"],
        "description": "약한 원근 왜곡",
    },
    {
        "id": "Y08", "name": "yolo_mosaic", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic"], "required_args": ["mosaic"],
        "description": "여러 이미지를 한 학습 장면에 구성",
    },
    {
        "id": "Y09", "name": "yolo_mixup", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mixup"], "required_args": ["mixup"],
        "description": "두 이미지와 label을 blending",
    },
    {
        "id": "Y10", "name": "yolo_cutmix", "family": "yolo_single",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["cutmix"], "required_args": ["cutmix"],
        "description": "다른 이미지의 직사각형 영역을 붙여 occlusion 생성",
    },

    # YOLO combinations
    {
        "id": "Y11", "name": "yolo_hsv_flip", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["hsv_flip"], "required_args": ["hsv_h", "fliplr"],
        "description": "HSV + 좌우 반전",
    },
    {
        "id": "Y12", "name": "yolo_geometry_combo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["geo_combo"], "required_args": ["degrees", "translate", "scale"],
        "description": "회전/이동/scale/shear/perspective/flip 조합",
    },
    {
        "id": "Y13", "name": "yolo_mosaic_hsv_geo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic_hsv_geo"], "required_args": ["mosaic", "hsv_h", "degrees"],
        "description": "Mosaic + HSV + 약한 geometry",
    },
    {
        "id": "Y14", "name": "yolo_mosaic_mixup_cutmix", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["mosaic_mixup_cutmix"], "required_args": ["mosaic", "mixup", "cutmix"],
        "description": "세 가지 multi-image augmentation 조합",
    },
    {
        "id": "Y15", "name": "yolo_balanced_combo", "family": "yolo_combo",
        "data_kind": "processed", "cv_policy": None,
        "aug": YOLO_AUG_CONFIGS["balanced_combo"], "required_args": ["hsv_h", "degrees", "mosaic"],
        "description": "색/기하/multi-image를 모두 약하게 섞은 조합",
    },

    # OpenCV static augmentation
    *[
        {
            "id": f"C{index:02d}",
            "name": f"opencv_{policy_name}",
            "family": "opencv_single" if policy_name not in {"photometric_combo", "geometric_combo", "mixed_combo"} else "opencv_combo",
            "data_kind": "opencv",
            "cv_policy": policy_name,
            "aug": NO_AUG,
            "required_args": [],
            "description": f"OpenCV static augmentation: {policy_name}",
        }
        for index, policy_name in enumerate(CV_POLICIES.keys(), start=1)
    ],

    # Hybrid: static OpenCV + online YOLO
    {
        "id": "H01", "name": "cv_photo_plus_yolo_geo", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "photometric_combo",
        "aug": YOLO_AUG_CONFIGS["geo_combo"], "required_args": ["degrees", "translate", "scale"],
        "description": "OpenCV photometric + YOLO online geometry",
    },
    {
        "id": "H02", "name": "cv_geo_plus_yolo_hsv", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "geometric_combo",
        "aug": YOLO_AUG_CONFIGS["hsv"], "required_args": ["hsv_h", "hsv_s", "hsv_v"],
        "description": "OpenCV geometry + YOLO online HSV",
    },
    {
        "id": "H03", "name": "cv_mixed_plus_yolo_light", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "mixed_combo",
        "aug": with_no_aug(
            hsv_h=0.010, hsv_s=0.30, hsv_v=0.25,
            degrees=5.0, translate=0.04, scale=0.12,
            fliplr=0.30, mosaic=0.30, close_mosaic=3,
        ),
        "required_args": ["hsv_h", "degrees", "mosaic"],
        "description": "OpenCV mixed + 약한 YOLO online augmentation",
    },
    {
        "id": "H04", "name": "cv_jpeg_plus_yolo_hsv_geo", "family": "hybrid",
        "data_kind": "opencv", "cv_policy": "jpeg_compression",
        "aug": with_no_aug(
            hsv_h=0.012, hsv_s=0.40, hsv_v=0.30,
            degrees=7.0, translate=0.05, scale=0.15, fliplr=0.40,
        ),
        "required_args": ["hsv_h", "degrees", "scale"],
        "description": "업로드 압축 품질 저하 + YOLO 색/기하 변화",
    },
]

# hp_mode를 명시하지 않은 실험은 모두 Optuna tuned를 사용합니다.
for experiment in EXPERIMENTS:
    experiment.setdefault("hp_mode", "tuned")

if SMOKE_TEST:
    smoke_ids = {"B00", "B01", "B02", "B03", "Y01", "C01", "H01"}
    EXPERIMENTS = [experiment for experiment in EXPERIMENTS if experiment["id"] in smoke_ids]

planned_experiments_df = pd.DataFrame([
    {
        "id": experiment["id"],
        "name": experiment["name"],
        "family": experiment["family"],
        "hp_mode": experiment["hp_mode"],
        "data_kind": experiment["data_kind"],
        "cv_policy": experiment["cv_policy"],
        "description": experiment["description"],
    }
    for experiment in EXPERIMENTS
])

display(planned_experiments_df)
print("총 본 실험 수:", len(EXPERIMENTS))

## 10. 공통 함수

In [ ]:
def missing_required_args(experiment):
    if not DEFAULT_CFG_DICT:
        return []

    supported = set(DEFAULT_CFG_DICT.keys())
    return [
        argument
        for argument in experiment.get("required_args", [])
        if argument not in supported
    ]


compatibility_rows = []

for experiment in EXPERIMENTS:
    missing = missing_required_args(experiment)
    compatibility_rows.append({
        "id": experiment["id"],
        "name": experiment["name"],
        "supported": len(missing) == 0,
        "missing_required_args": ", ".join(missing),
    })

compatibility_df = pd.DataFrame(compatibility_rows)
display(compatibility_df)

In [ ]:
def dataset_yaml_for_experiment(experiment, seed):
    if experiment["data_kind"] == "processed":
        return RUNTIME_DATA_YAML

    if experiment["data_kind"] == "opencv":
        return build_opencv_dataset(
            experiment["cv_policy"],
            seed=seed,
            apply_probability=CV_APPLY_PROBABILITY,
            overwrite=False,
        )

    raise ValueError(f"알 수 없는 data_kind: {experiment['data_kind']}")

In [ ]:
def extract_metrics(metrics):
    box = metrics.box

    precision = float(getattr(box, "mp", np.nan))
    recall = float(getattr(box, "mr", np.nan))

    if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    speed = getattr(metrics, "speed", {}) or {}

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(box.map50),
        "mAP75": float(box.map75),
        "mAP50_95": float(box.map),
        "inference_ms_per_image": speed.get("inference", np.nan),
    }

In [ ]:
def save_per_class_metrics(metrics, experiment_id, experiment_name, seed):
    maps = np.asarray(metrics.box.maps, dtype=float)

    per_class = pd.DataFrame({
        "class_id": range(len(maps)),
        "class_name": [CLASS_NAMES.get(index, f"class_{index}") for index in range(len(maps))],
        "mAP50_95": maps,
    })

    if len(class_support_df):
        support_cols = [col for col in ["class_name", "train", "val"] if col in class_support_df.columns]
        per_class = per_class.merge(
            class_support_df[support_cols],
            on="class_name",
            how="left",
        )

    path = PER_CLASS_DIR / f"{experiment_id}_{experiment_name}_seed{seed}.csv"
    per_class.to_csv(path, index=False, encoding="utf-8-sig")
    return path

# 11. Optuna 탐색 공간

`optimizer`에 따라 `lr0` 범위를 나눕니다. Optuna는 파라미터 이름이 같으면
분포도 같아야 하므로, 이름을 `lr0_adamw` / `lr0_sgd`로 분리하고
실제로 쓰인 값은 `lr0` user attribute로 따로 기록합니다.

| 파라미터 | 범위 | 01번과 달라진 점 |
|---|---|---|
| optimizer | AdamW / SGD | 동일 |
| lr0 (AdamW) | 1e-4 ~ 3e-3 | **신규 분리** |
| lr0 (SGD) | 1e-3 ~ 2e-2 | **신규 분리**, 하한 상향 |
| lrf | 0.01 ~ 1.0 | 동일 (중요도 1위) |
| momentum | 0.80 ~ 0.98 | 하한 0.70 -> 0.80 |
| weight_decay | 1e-6 ~ 1e-3 | 동일 |
| warmup_epochs | 0 ~ 5 | 동일 |
| cos_lr | True / False | 동일 |


In [ ]:
def suggest_core_hyperparameters(trial):
    optimizer = trial.suggest_categorical(
        "optimizer",
        ["AdamW", "SGD"],
    )

    # 01번 교훈: optimizer마다 적정 learning rate 대역이 다릅니다.
    # AdamW에 SGD용 lr을 주면 학습이 무너집니다.
    if optimizer == "AdamW":
        lr0 = trial.suggest_float("lr0_adamw", 1e-4, 3e-3, log=True)
    else:
        lr0 = trial.suggest_float("lr0_sgd", 1e-3, 2e-2, log=True)

    trial.set_user_attr("lr0", float(lr0))

    return {
        "optimizer": optimizer,
        "lr0": lr0,
        "lrf": trial.suggest_float("lrf", 0.01, 1.0, log=True),
        "momentum": trial.suggest_float("momentum", 0.80, 0.98),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "warmup_epochs": trial.suggest_float("warmup_epochs", 0.0, 5.0),
        "cos_lr": trial.suggest_categorical("cos_lr", [False, True]),
    }


# 12. Pruning 콜백

매 epoch의 Validation mAP50-95를 Optuna에 보고하고,
다른 trial보다 확실히 뒤처지면 학습을 조기 종료합니다.

In [ ]:
def find_map50_95_in_trainer_metrics(metrics_dict):
    if not metrics_dict:
        return None

    for key, value in metrics_dict.items():
        if "map50-95" in str(key).lower().replace(" ", ""):
            try:
                return float(value)
            except (TypeError, ValueError):
                return None

    return None


def make_optuna_pruning_callback(trial, max_epochs):
    state = {"pruned": False, "last_step": -1}

    def callback(trainer):
        step = int(getattr(trainer, "epoch", -1))

        # 같은 step 중복 보고와 최종 평가 시점 호출을 걸러냅니다.
        if step <= state["last_step"] or step >= max_epochs:
            return

        value = find_map50_95_in_trainer_metrics(getattr(trainer, "metrics", {}))

        if value is None or not np.isfinite(value):
            return

        state["last_step"] = step
        trial.report(value, step=step)

        if trial.should_prune():
            state["pruned"] = True
            trainer.stop = True

    return callback, state


# 13. trial 하나를 수행하는 목적 함수

모든 trial은 **증강을 전부 끈 상태**(`NO_AUG`)에서 돌립니다.
02번의 `B00` / `B03`과 정확히 같은 조건이라 바로 비교할 수 있습니다.

In [ ]:
def cleanup_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def objective(trial):
    cleanup_memory()

    params = suggest_core_hyperparameters(trial)
    run_name = f"trial_{trial.number:04d}"

    model = YOLO(MODEL_NAME)

    pruning_callback, pruning_state = make_optuna_pruning_callback(
        trial,
        max_epochs=TUNE_EPOCHS,
    )

    model.add_callback("on_fit_epoch_end", pruning_callback)

    started = time.perf_counter()

    try:
        model.train(
            data=str(RUNTIME_DATA_YAML),
            epochs=TUNE_EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            patience=TUNE_PATIENCE,
            device=DEVICE,
            workers=WORKERS,
            seed=SEED,
            deterministic=True,
            project=str(TUNE_RUNS_DIR),
            name=run_name,
            exist_ok=True,
            save=True,
            plots=False,
            verbose=False,
            **NO_AUG,
            **params,
        )

        train_minutes = (time.perf_counter() - started) / 60.0
        save_dir = Path(model.trainer.save_dir)

        if pruning_state["pruned"]:
            if not KEEP_TUNE_RUNS:
                shutil.rmtree(save_dir, ignore_errors=True)

            raise optuna.TrialPruned(f"epoch {pruning_state['last_step'] + 1} 부근에서 중단")

        best_pt = save_dir / "weights" / "best.pt"

        if not best_pt.exists():
            raise FileNotFoundError(best_pt)

        best_model = YOLO(str(best_pt))

        val_metrics = best_model.val(
            data=str(RUNTIME_DATA_YAML),
            split="val",
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            workers=WORKERS,
            plots=False,
            verbose=False,
        )

        metric_dict = extract_metrics(val_metrics)

        for key, value in metric_dict.items():
            if value is not None and np.isfinite(value):
                trial.set_user_attr(key, float(value))

        trial.set_user_attr("train_minutes", float(train_minutes))

        objective_value = metric_dict["mAP50_95"]

        if not KEEP_TUNE_RUNS:
            shutil.rmtree(save_dir, ignore_errors=True)

        del best_model
        del model
        cleanup_memory()

        return objective_value

    except optuna.TrialPruned:
        del model
        cleanup_memory()
        raise

    except torch.cuda.OutOfMemoryError:
        cleanup_memory()
        raise optuna.TrialPruned("CUDA OOM")

    except Exception:
        cleanup_memory()
        raise


# 14. Study 생성과 좋은 출발점 주입

`enqueue_trial`로 두 조합을 첫 trial로 예약합니다.

1. **auto 근사** — 00번 baseline에서 `auto`가 실제로 고른 것이 AdamW였고,
   17개 카테고리에서 가장 좋았던 조건입니다.
2. **01번 1위** — 86개 카테고리 최고 조합. 여기서도 좋은지 직접 확인합니다.

## 중단하고 다시 실행해도 됩니다

이 셀은 실행할 때마다 다음을 합니다.

1. 이전 세션이 끊겨 `RUNNING` 으로 남은 trial 을 `FAIL` 로 정리
2. 두 출발점 중 **아직 제대로 시도되지 않은 것만** 다시 예약

`FAIL` 은 목표 trial 수에 포함되지 않으므로 그만큼 다시 수행됩니다.
따라서 학습 도중에 멈추더라도 **DB 나 폴더를 손으로 지울 필요가 없습니다.**
그냥 커널을 재시작하고 처음부터 실행하면 됩니다.

In [ ]:
STUDY_NAME = "yolo26n_17class_hpo"

storage_url = f"sqlite:///{STUDY_DB.as_posix()}"

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=storage_url,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=4,
        n_warmup_steps=5,
        interval_steps=1,
    ),
    load_if_exists=True,
)

WARM_START_TRIALS = [
    {
        # optimizer="auto"가 고르는 것과 비슷한 AdamW 설정
        "optimizer": "AdamW",
        "lr0_adamw": 0.00125,
        "lrf": 0.01,
        "momentum": 0.90,
        "weight_decay": 0.0005,
        "warmup_epochs": 3.0,
        "cos_lr": False,
    },
    {
        # 01번(86개 카테고리) 1위 조합
        "optimizer": "SGD",
        "lr0_sgd": 0.00754,
        "lrf": 0.46951,
        "momentum": 0.91358,
        "weight_decay": 0.00006,
        "warmup_epochs": 4.48054,
        "cos_lr": True,
    },
]

def mark_stale_running_trials_failed(study):
    """이전 세션이 강제 종료되면 trial 이 RUNNING 인 채로 DB 에 남습니다.

    그대로 두면 Optuna 가 계속 '진행 중'으로 보므로 FAIL 로 정리합니다.
    FAIL 은 목표 trial 수에 포함되지 않으니 그만큼 다시 수행됩니다."""
    running = [t for t in study.trials if t.state == TrialState.RUNNING]

    for trial in running:
        try:
            study.tell(trial.number, state=TrialState.FAIL, skip_if_finished=True)
            print(f"끊긴 Trial {trial.number} -> FAIL 처리")
        except Exception as error:
            print(f"Trial {trial.number} 상태 변경 실패:", repr(error))


def warm_start_settled(study, params):
    """이 출발점이 이미 대기 중이거나 정상 수행된 적이 있으면 True.

    enqueue_trial 로 예약한 값은 trial.system_attrs["fixed_params"] 에 남고
    FAIL 이 되어도 지워지지 않습니다. 그래서 FAIL 은 건너뛰고 비교하면
    '아직 제대로 시도되지 않은 출발점'만 골라낼 수 있습니다."""
    for trial in study.trials:
        if trial.state == TrialState.FAIL:
            continue

        recorded = trial.system_attrs.get("fixed_params") or trial.params

        if recorded and all(recorded.get(key) == value for key, value in params.items()):
            return True

    return False


# 먼저 끊긴 trial 을 정리해야, 그 trial 이 물고 있던 출발점을 다시 예약할 수 있습니다.
mark_stale_running_trials_failed(study)

pending = [p for p in WARM_START_TRIALS if not warm_start_settled(study, p)]

for params in pending:
    study.enqueue_trial(params)

if pending:
    print(f"출발점 {len(pending)}개를 예약했습니다.")
else:
    print("출발점은 이미 모두 시도되었습니다.")

if study.trials:
    print(f"기존 Study 를 이어서 사용합니다. 현재 trial {len(study.trials)}개")

print("Study :", STUDY_NAME)
print("DB    :", STUDY_DB)


# 15. Optuna 실행

`COMPLETE + PRUNED` 기준 총 `TARGET_TOTAL_TRIALS`개가 될 때까지만 돌립니다.
중간에 끊겨도 DB에 저장된 trial을 세고 이어서 진행합니다.

In [ ]:
RUN_OPTUNA = True


def valid_trial_count(study):
    return sum(
        trial.state in {TrialState.COMPLETE, TrialState.PRUNED}
        for trial in study.trials
    )


# 끊긴 trial 정리와 출발점 재예약은 앞의 Study 생성 셀에서 이미 끝났습니다.

print(f"현재 유효 trial: {valid_trial_count(study)} / {TARGET_TOTAL_TRIALS}")

if RUN_OPTUNA:
    while valid_trial_count(study) < TARGET_TOTAL_TRIALS:
        print("=" * 90)
        print(f"Optuna 진행: {valid_trial_count(study)}/{TARGET_TOTAL_TRIALS}")
        print("=" * 90)

        # 한 번에 하나씩 수행해야 중간에 끊겨도 정확히 이어집니다.
        study.optimize(objective, n_trials=1, n_jobs=1, gc_after_trial=True)

else:
    print("RUN_OPTUNA=False")

print()
print("최종 유효 trial:", valid_trial_count(study), "/", TARGET_TOTAL_TRIALS)


## 16. Trial 결과 정리

In [ ]:
trial_rows = []

for trial in study.trials:
    row = {
        "trial": trial.number,
        "state": trial.state.name,
        "mAP50_95": trial.value,
        **{f"param_{k}": v for k, v in trial.params.items()},
        **dict(trial.user_attrs),
    }
    trial_rows.append(row)

trials_df = pd.DataFrame(trial_rows)

if "mAP50_95" in trials_df.columns:
    trials_df = trials_df.sort_values("mAP50_95", ascending=False, na_position="last")

TRIALS_CSV = OPTUNA_REPORT_DIR / "trials.csv"
trials_df.to_csv(TRIALS_CSV, index=False, encoding="utf-8-sig")

display(trials_df)
print("Saved:", TRIALS_CSV)

completed_df = trials_df[
    trials_df["state"].eq("COMPLETE") & trials_df["mAP50_95"].notna()
].copy()

print("COMPLETE trial:", len(completed_df))


## 17. 탐색 과정과 파라미터 중요도

In [ ]:
set_korean_font()

by_trial = completed_df.sort_values("trial")

if len(by_trial):
    by_trial = by_trial.assign(best_so_far=by_trial["mAP50_95"].cummax())

    plt.figure(figsize=(10, 5))
    plt.plot(by_trial["trial"], by_trial["mAP50_95"], marker="o", alpha=0.6, label="trial 성능")
    plt.plot(by_trial["trial"], by_trial["best_so_far"], linewidth=2, label="누적 최고")
    plt.xlabel("Trial 번호")
    plt.ylabel("Validation mAP50-95")
    plt.title("Optuna 탐색 진행 (17개 카테고리)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    figure_path = OPTUNA_FIG_DIR / "optimization_history.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)

if len(completed_df) >= 4:
    try:
        importance = optuna.importance.get_param_importances(study)

        importance_df = pd.DataFrame({
            "parameter": list(importance.keys()),
            "importance": list(importance.values()),
        })

        importance_df.to_csv(
            OPTUNA_REPORT_DIR / "parameter_importance.csv",
            index=False,
            encoding="utf-8-sig",
        )

        display(importance_df.round(4))

        plot_df = importance_df.sort_values("importance")

        plt.figure(figsize=(9, 5))
        plt.barh(plot_df["parameter"], plot_df["importance"])
        plt.xlabel("추정 중요도")
        plt.title("하이퍼파라미터 중요도")
        plt.tight_layout()

        figure_path = OPTUNA_FIG_DIR / "parameter_importance.png"
        plt.savefig(figure_path, dpi=160, bbox_inches="tight")
        plt.show()
        print("Saved:", figure_path)

    except Exception as error:
        print("중요도 계산 생략:", error)

else:
    print("COMPLETE trial이 적어 중요도 계산을 생략합니다.")


# 18. 상위 3개 후보를 seed 3개로 재검증

단일 run 1등은 seed 운일 수 있으므로 상위 3개를 각각 3개 seed로 학습합니다.
이 단계도 **10 epoch**이며 증강은 전부 꺼진 상태입니다.

In [ ]:
MULTISEED_CSV = OPTUNA_REPORT_DIR / "candidate_multiseed.csv"


def params_from_trial(trial):
    """조건부 파라미터(lr0_adamw / lr0_sgd)를 실제 학습 인자로 되돌립니다."""
    raw = dict(trial.params)
    optimizer = raw.pop("optimizer")
    lr0 = raw.pop("lr0_adamw", None)

    if lr0 is None:
        lr0 = raw.pop("lr0_sgd")
    else:
        raw.pop("lr0_sgd", None)

    return {"optimizer": optimizer, "lr0": float(lr0), **raw}


def train_candidate(params, seed, policy_name, epochs):
    cleanup_memory()

    model = YOLO(MODEL_NAME)
    started = time.perf_counter()

    model.train(
        data=str(RUNTIME_DATA_YAML),
        epochs=epochs,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        device=DEVICE,
        workers=WORKERS,
        seed=seed,
        deterministic=True,
        project=str(RUNS_DIR),
        name=f"{policy_name}_seed{seed}_e{epochs}",
        exist_ok=True,
        save=True,
        plots=True,
        verbose=False,
        **NO_AUG,
        **params,
    )

    train_minutes = (time.perf_counter() - started) / 60.0
    save_dir = Path(model.trainer.save_dir)
    best_pt = save_dir / "weights" / "best.pt"

    if not best_pt.exists():
        raise FileNotFoundError(best_pt)

    best_model = YOLO(str(best_pt))

    val_metrics = best_model.val(
        data=str(RUNTIME_DATA_YAML),
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        plots=False,
        verbose=False,
    )

    row = {
        "policy": policy_name,
        "seed": seed,
        "requested_epochs": epochs,
        "train_minutes": train_minutes,
        "best_pt": str(best_pt),
        **extract_metrics(val_metrics),
    }

    del best_model
    del model
    cleanup_memory()

    return row


if not len(completed_df):
    raise RuntimeError("COMPLETE trial이 없습니다. Optuna 셀을 먼저 실행하세요.")

CANDIDATE_TRIALS = [
    int(number)
    for number in completed_df.sort_values("mAP50_95", ascending=False)
    .head(TOP_K_FOR_MULTI_SEED)["trial"]
]

SINGLE_RUN_BEST_TRIAL = CANDIDATE_TRIALS[0]

print("multi-seed 대상 trial:", CANDIDATE_TRIALS)
print("단일 run 1위 (참고)  :", SINGLE_RUN_BEST_TRIAL)

multiseed_rows = []

if MULTISEED_CSV.exists():
    multiseed_rows = pd.read_csv(MULTISEED_CSV).to_dict("records")
    print("기존 결과", len(multiseed_rows), "건을 이어받았습니다.")


def already_done(rows, policy_name, seed, epochs):
    return any(
        row.get("policy") == policy_name
        and int(row.get("seed", -1)) == seed
        and int(row.get("requested_epochs", -1)) == epochs
        for row in rows
    )


for trial_number in CANDIDATE_TRIALS:
    trial = study.trials[trial_number]
    params = params_from_trial(trial)
    policy_name = f"optuna17_trial{trial_number:04d}"

    for seed in MULTI_SEEDS:
        if already_done(multiseed_rows, policy_name, seed, TUNE_EPOCHS):
            print(f"skip: {policy_name} seed={seed}")
            continue

        print("=" * 90)
        print(f"{policy_name} / seed {seed} / {TUNE_EPOCHS} epoch")
        print("params:", params)

        row = train_candidate(params, seed, policy_name, TUNE_EPOCHS)
        row["trial"] = trial_number
        row["single_run_mAP50_95"] = trial.value
        multiseed_rows.append(row)

        pd.DataFrame(multiseed_rows).to_csv(
            MULTISEED_CSV, index=False, encoding="utf-8-sig"
        )

multiseed_df = pd.DataFrame(multiseed_rows)
display(multiseed_df)


## 19. 후보별 평균과 최종 선택

In [ ]:
set_korean_font()

candidate_summary = (
    multiseed_df
    .groupby(["trial", "policy"])
    .agg(
        runs=("mAP50_95", "count"),
        mAP50_95_mean=("mAP50_95", "mean"),
        mAP50_95_std=("mAP50_95", "std"),
        recall_mean=("recall", "mean"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        single_run_mAP50_95=("single_run_mAP50_95", "max"),
    )
    .reset_index()
    .sort_values(["mAP50_95_mean", "recall_mean"], ascending=False)
)

display(candidate_summary.round(4))

candidate_summary.to_csv(
    OPTUNA_REPORT_DIR / "candidate_multiseed_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

plot_df = candidate_summary.sort_values("mAP50_95_mean")
positions = np.arange(len(plot_df))

plt.figure(figsize=(10, 5))
plt.barh(
    positions,
    plot_df["mAP50_95_mean"],
    xerr=plot_df["mAP50_95_std"].fillna(0),
    capsize=4,
    label="3-seed 평균",
)
plt.scatter(
    plot_df["single_run_mAP50_95"],
    positions,
    color="black",
    zorder=3,
    label="단일 run",
)
plt.yticks(positions, [f"trial {int(t)}" for t in plot_df["trial"]])
plt.xlabel("Validation mAP50-95")
plt.title("상위 후보: 단일 run vs 3-seed 평균")
plt.legend()
plt.tight_layout()

figure_path = OPTUNA_FIG_DIR / "candidate_multiseed.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)

FINAL_TRIAL_NUMBER = int(candidate_summary.iloc[0]["trial"])
final_trial = study.trials[FINAL_TRIAL_NUMBER]
FINAL_PARAMS = params_from_trial(final_trial)

print()
print("최종 선택 trial:", FINAL_TRIAL_NUMBER)
for key, value in FINAL_PARAMS.items():
    print(f"  {key:>15}: {value}")

if FINAL_TRIAL_NUMBER != SINGLE_RUN_BEST_TRIAL:
    print()
    print(f"참고: 단일 run 1위는 trial {SINGLE_RUN_BEST_TRIAL}였지만 "
          f"3-seed 평균으로는 trial {FINAL_TRIAL_NUMBER}가 선택되었습니다.")


# 20. 최종 후보를 15 epoch으로 한 번 더 학습

02번의 `B00`(15 epoch, 증강 OFF, auto)과 직접 비교할 수 있는 조건입니다.
여기서 나온 값이 **17개 카테고리에서의 진짜 하이퍼파라미터 효과**입니다.

In [ ]:
FINAL_POLICY = f"optuna17_best_trial{FINAL_TRIAL_NUMBER:04d}"

final_rows = [
    row for row in multiseed_rows
    if row.get("policy") == FINAL_POLICY
    and int(row.get("requested_epochs", -1)) == FINAL_COMPARE_EPOCHS
]

if final_rows:
    print("이미 15 epoch 결과가 있습니다.")
    final_row = final_rows[0]

else:
    final_row = train_candidate(
        FINAL_PARAMS,
        SEED,
        FINAL_POLICY,
        FINAL_COMPARE_EPOCHS,
    )
    final_row["trial"] = FINAL_TRIAL_NUMBER
    final_row["single_run_mAP50_95"] = final_trial.value

    multiseed_rows.append(final_row)
    pd.DataFrame(multiseed_rows).to_csv(
        MULTISEED_CSV, index=False, encoding="utf-8-sig"
    )

display(pd.DataFrame([final_row]).round(4))


## 21. 새 하이퍼파라미터 저장

`selected_trial.json`을 05번 폴더에 저장합니다.
06번 노트북은 **01번보다 이 파일을 먼저** 찾습니다.

In [ ]:
# 어떤 데이터로 탐색했는지 함께 기록합니다. 06번이 이 값을 보고 경고할 수 있습니다.
DATASET_FINGERPRINT = f"17class_train{len(train_images)}_val{len(val_images)}"

SELECTED_JSON = OPTUNA_REPORT_DIR / "selected_trial.json"

selected_record = {
    "selected_trial": FINAL_TRIAL_NUMBER,
    "dataset": DATASET_FINGERPRINT,
    "selection_rule": (
        f"top {TOP_K_FOR_MULTI_SEED} by single-run mAP50-95 at {TUNE_EPOCHS} epoch, "
        f"re-trained with seeds {MULTI_SEEDS}, highest mean mAP50-95"
    ),
    "objective_mAP50_95": float(candidate_summary.iloc[0]["mAP50_95_mean"]),
    "final_15epoch_mAP50_95": float(final_row["mAP50_95"]),
    "metrics": {
        key: (None if pd.isna(final_row.get(key)) else float(final_row.get(key)))
        for key in ["precision", "recall", "f1", "mAP50", "mAP50_95"]
    },
    "params": FINAL_PARAMS,
    "fixed": {
        "model": MODEL_NAME,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "tune_epochs": TUNE_EPOCHS,
        "final_epochs": FINAL_COMPARE_EPOCHS,
    },
}

with open(SELECTED_JSON, "w", encoding="utf-8") as file:
    json.dump(selected_record, file, ensure_ascii=False, indent=2)

print(json.dumps(selected_record, ensure_ascii=False, indent=2))
print()
print("Saved:", SELECTED_JSON)

# 이후 증강 실험에서 사용할 값
TUNED_PARAMS = dict(FINAL_PARAMS)
HP_SOURCE = "optuna_tuned_17class"

print()
print("증강 실험에 적용할 하이퍼파라미터:", TUNED_PARAMS)


## 21-1. 하이퍼파라미터 상위 2개 저장

06번은 **하이퍼파라미터 top2 x 증강 top2**를 교차로 실험합니다.
그래서 1위뿐 아니라 2위 조합도 함께 저장해 둡니다.

multi-seed 평균 순위를 기준으로 하며, 각각 `hp1` / `hp2` 라는 이름을 붙입니다.

In [ ]:
TOP2_HP_JSON = OPTUNA_REPORT_DIR / "top2_hyperparameters.json"

top2_hp_records = []

for rank, (_, row) in enumerate(candidate_summary.head(2).iterrows(), start=1):
    trial_number = int(row["trial"])

    top2_hp_records.append({
        "rank": rank,
        "label": f"hp{rank}",
        "trial": trial_number,
        "multiseed_mAP50_95_mean": float(row["mAP50_95_mean"]),
        "multiseed_mAP50_95_std": (
            None if pd.isna(row["mAP50_95_std"]) else float(row["mAP50_95_std"])
        ),
        "params": params_from_trial(study.trials[trial_number]),
    })

top2_hp_payload = {
    "dataset": DATASET_FINGERPRINT,
    "selection_rule": (
        f"multi-seed({MULTI_SEEDS}) 평균 mAP50-95 상위 2개, {TUNE_EPOCHS} epoch 기준"
    ),
    "fixed": {
        "model": MODEL_NAME,
        "imgsz": IMGSZ,
        "batch": BATCH,
    },
    "hyperparameters": top2_hp_records,
}

with open(TOP2_HP_JSON, "w", encoding="utf-8") as file:
    json.dump(top2_hp_payload, file, ensure_ascii=False, indent=2)

for record in top2_hp_records:
    print(f"[{record['label']}] trial {record['trial']} "
          f"평균 mAP50-95 {record['multiseed_mAP50_95_mean']:.4f}")
    for key, value in record["params"].items():
        print(f"      {key:>15}: {value}")
    print()

if len(top2_hp_records) < 2:
    print("주의: 후보가 2개 미만입니다. 06번 교차 실험이 줄어듭니다.")

print("Saved:", TOP2_HP_JSON)


# 22. 증강 후보 정하기

`USE_ALL_AUGMENTATIONS = True` 이면 **02번 카탈로그의 증강 33개를 전부** 재실험합니다.
`False` 로 두면 02번 순위 상위 `AUG_SHORTLIST_TOP_N` 개만 돌립니다.

## 왜 전부 돌리나

02번의 증강 순위는 `close_mosaic` 버그의 영향을 받아 신뢰하기 어렵습니다.
mosaic 계열 7개(`Y08` `Y09` `Y10` `Y13` `Y14` `Y15` `H03`)가 전 구간에서 꺼진 채
학습되어 **서로 완전히 같은 점수**가 나왔기 때문입니다.

이제 `close_mosaic` 를 고쳤으므로, 그 기법들은 **이번에 처음으로 제대로 평가**됩니다.
좁게 추리는 대신 전부 다시 보는 편이 맞습니다. epoch 이 10 이라 감당 가능합니다.

여기에 **대조군 4개를 반드시 함께** 돌립니다. 새 하이퍼파라미터 기준의
2x2 대조가 있어야 증강 효과와 하이퍼파라미터 효과를 분리할 수 있습니다.

|                | optimizer auto | 새 tuned |
|----------------|----------------|----------|
| 증강 전부 OFF   | `B00`          | `B03`    |
| YOLO 기본 증강  | `B01`          | `B02`    |

즉 이 단계에서 도는 실험은 **대조군 4개 + 증강 33개 = 37개**입니다.


In [ ]:
NB02_SUMMARY_DIR = (
    PROJECT_ROOT / "../models/yolo/02_experiment_augmentation/report/summary"
).resolve()

CONTROL_IDS = ["B00", "B01", "B02", "B03"]

nb02_results_path = NB02_SUMMARY_DIR / "experiment_results.csv"

if not nb02_results_path.exists():
    raise FileNotFoundError(
        f"02번 결과를 찾지 못했습니다: {nb02_results_path}\n"
        "02번을 먼저 실행하거나 AUG_SHORTLIST_IDS에 직접 지정하세요."
    )

nb02_df = pd.read_csv(nb02_results_path)

nb02_screen = nb02_df[
    nb02_df["status"].eq("OK")
    & nb02_df["requested_epochs"].eq(EPOCHS)
    & ~nb02_df["id"].isin(CONTROL_IDS)
]

nb02_rank = (
    nb02_screen
    .groupby(["id", "name"])["mAP50_95"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print(f"02번 증강 순위 (상위 {AUG_SHORTLIST_TOP_N}개)")
display(nb02_rank.head(AUG_SHORTLIST_TOP_N).round(4))

# 직접 고르고 싶으면 여기에 id를 적으세요. 예: ["Y05", "Y11", "C06"]
AUG_SHORTLIST_IDS_OVERRIDE = None

catalog_aug_ids = [
    experiment["id"]
    for experiment in EXPERIMENTS
    if experiment["id"] not in CONTROL_IDS
]

if AUG_SHORTLIST_IDS_OVERRIDE:
    AUG_SHORTLIST_IDS = list(AUG_SHORTLIST_IDS_OVERRIDE)

elif USE_ALL_AUGMENTATIONS:
    # 02번 순위대로 정렬해 두면 결과 표가 읽기 편합니다.
    # (02번 mosaic 계열 순위는 close_mosaic 버그의 영향을 받았으므로
    #  여기서는 '표시 순서' 용도로만 씁니다.)
    ranked = [key for key in nb02_rank["id"].tolist() if key in catalog_aug_ids]
    AUG_SHORTLIST_IDS = ranked + [
        key for key in catalog_aug_ids if key not in ranked
    ]

else:
    AUG_SHORTLIST_IDS = nb02_rank.head(AUG_SHORTLIST_TOP_N)["id"].tolist()

print()
print("재실험할 증강:", AUG_SHORTLIST_IDS)


## 23. 재실험 대상 구성

In [ ]:
catalog_by_id = {experiment["id"]: experiment for experiment in EXPERIMENTS}

missing_ids = [key for key in AUG_SHORTLIST_IDS if key not in catalog_by_id]

if missing_ids:
    raise KeyError(f"카탈로그에 없는 ID: {missing_ids}")

retune_experiments = []

for key in CONTROL_IDS:
    experiment = dict(catalog_by_id[key])
    retune_experiments.append(experiment)

for key in AUG_SHORTLIST_IDS:
    experiment = dict(catalog_by_id[key])
    experiment["hp_mode"] = "tuned"
    retune_experiments.append(experiment)

EXPERIMENTS = retune_experiments

display(
    pd.DataFrame(EXPERIMENTS)[
        ["id", "name", "family", "data_kind", "cv_policy", "hp_mode", "description"]
    ]
)

print()
print(f"총 {len(EXPERIMENTS)}개 x {EPOCHS} epoch")


## 24. 학습 실행 함수

02번과 같은 코드입니다. `TUNED_PARAMS`만 새 값으로 바뀌었습니다.

In [ ]:
RESULTS_CSV = SUMMARY_DIR / "experiment_results.csv"


def hp_source_for(experiment):
    """이 실험이 실제로 어떤 하이퍼파라미터로 학습되는지 반환합니다."""
    if experiment.get("hp_mode", "tuned") == "tuned" and TUNED_PARAMS:
        return "optuna_tuned"

    return "auto"


def train_one_experiment(experiment, seed=SEED, epochs=None):
    epochs = EPOCHS if epochs is None else epochs

    missing = missing_required_args(experiment)

    if missing:
        return {
            "status": "SKIPPED_UNSUPPORTED",
            "id": experiment["id"],
            "name": experiment["name"],
            "family": experiment["family"],
            "seed": seed,
            "requested_epochs": epochs,
            "hp_mode": experiment.get("hp_mode", "tuned"),
            "hp_source": hp_source_for(experiment),
            "error": f"unsupported args: {missing}",
        }

    dataset_yaml = dataset_yaml_for_experiment(experiment, seed)
    # epoch 수가 다르면 서로 덮어쓰지 않도록 run 이름을 구분합니다.
    run_name = f"{experiment['id']}_{experiment['name']}_seed{seed}_e{epochs}"

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 매우 중요: 모든 실험을 같은 pretrained weight에서 새로 시작합니다.
    model = YOLO(MODEL_NAME)

    train_kwargs = {
        "data": str(dataset_yaml),
        "epochs": epochs,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "patience": PATIENCE,
        "device": DEVICE,
        "workers": WORKERS,
        "seed": seed,
        "deterministic": True,
        # aug=None (B01/B02 처럼 YOLO 기본값을 쓰는 실험)은 Ultralytics 기본
        # close_mosaic=10 이 적용되어, epochs 가 10~15 면 mosaic 이 거의
        # 또는 전부 꺼집니다. 그래서 여기서 기본값을 깔아 둡니다.
        # (aug 딕셔너리에 close_mosaic 이 있으면 아래 update 에서 덮어씁니다)
        "close_mosaic": CLOSE_MOSAIC_EPOCHS,
        # TUNED_PARAMS가 있으면 아래에서 덮어씁니다.
        "optimizer": "auto",
        "amp": True,
        "cache": False,
        "project": str(RUNS_DIR),
        "name": run_name,
        "exist_ok": True,
        "plots": True,
        "verbose": True,
    }

    # hp_mode가 "tuned"인 실험에만 01번 Optuna 결과를 적용합니다.
    # B00 / B01은 "auto"이므로 위의 optimizer="auto" 기본값을 그대로 씁니다.
    experiment_hp_source = hp_source_for(experiment)

    if experiment_hp_source == "optuna_tuned":
        train_kwargs.update(TUNED_PARAMS)

    filtered_aug, unknown_aug = supported_train_args(experiment["aug"])

    if experiment["aug"] is not None:
        train_kwargs.update(filtered_aug)

    start_time = time.perf_counter()
    train_result = model.train(**train_kwargs)
    train_minutes = (time.perf_counter() - start_time) / 60.0

    save_dir = Path(train_result.save_dir)
    best_pt = save_dir / "weights" / "best.pt"

    if not best_pt.exists():
        raise FileNotFoundError(best_pt)

    history_csv = save_dir / "results.csv"
    actual_epochs = np.nan

    if history_csv.exists():
        history = pd.read_csv(history_csv)
        actual_epochs = len(history)

    best_model = YOLO(str(best_pt))

    val_metrics = best_model.val(
        data=str(dataset_yaml),
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        plots=False,
        verbose=False,
    )

    per_class_path = save_per_class_metrics(
        val_metrics,
        experiment["id"],
        experiment["name"],
        seed,
    )

    row = {
        "status": "OK",
        "id": experiment["id"],
        "name": experiment["name"],
        "family": experiment["family"],
        "description": experiment["description"],
        "seed": seed,
        "data_kind": experiment["data_kind"],
        "cv_policy": experiment["cv_policy"],
        "dataset_yaml": str(dataset_yaml),
        "requested_epochs": epochs,
        "actual_epochs": actual_epochs,
        "train_minutes": train_minutes,
        "best_pt": str(best_pt),
        "save_dir": str(save_dir),
        "per_class_csv": str(per_class_path),
        "unknown_filtered_aug_args": ",".join(unknown_aug),
        "hp_mode": experiment.get("hp_mode", "tuned"),
        "hp_source": experiment_hp_source,
        "hp_params": json.dumps(
            TUNED_PARAMS if experiment_hp_source == "optuna_tuned" else {},
            ensure_ascii=False,
        ),
        **extract_metrics(val_metrics),
    }

    # OpenCV static dataset은 용량이 클 수 있으므로 결과 보고서를 보존한 뒤 cache를 정리합니다.
    if experiment["data_kind"] == "opencv" and CLEANUP_CV_DATASET_AFTER_RUN:
        dataset_dir = Path(dataset_yaml).parent
        generation_report = dataset_dir / "generation_report.csv"

        if generation_report.exists():
            persistent_report = SUMMARY_DIR / f"opencv_generation_{experiment['cv_policy']}_seed{seed}.csv"
            shutil.copy2(generation_report, persistent_report)

        shutil.rmtree(dataset_dir, ignore_errors=True)

    del model
    del best_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row

In [ ]:
def load_existing_results():
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame()


def has_successful_run(existing_df, experiment, seed, epochs):
    """같은 (실험, seed, epoch, 하이퍼파라미터 설정) 조합이 이미 성공했는지 확인합니다."""
    if not len(existing_df):
        return False

    required_cols = {"status", "id", "seed"}
    if not required_cols.issubset(existing_df.columns):
        return False

    mask = (
        existing_df["status"].eq("OK")
        & existing_df["id"].eq(experiment["id"])
        & existing_df["seed"].eq(seed)
    )

    # epoch이 다르면 다른 실험입니다. (10 epoch 스크리닝 vs 15 epoch 최종 비교)
    if "requested_epochs" in existing_df.columns:
        mask = mask & existing_df["requested_epochs"].eq(epochs)
    else:
        return False

    # 하이퍼파라미터 설정이 바뀌면(auto <-> optuna_tuned) 같은 표에 둘 수 없으므로
    # 건너뛰지 않고 다시 학습합니다.
    if "hp_source" in existing_df.columns:
        previous_source = existing_df["hp_source"].fillna("auto")
    else:
        previous_source = pd.Series("auto", index=existing_df.index)

    mask = mask & previous_source.eq(hp_source_for(experiment))

    return bool(mask.any())


def run_experiment_list(experiments, seed=SEED, epochs=None):
    epochs = EPOCHS if epochs is None else epochs

    existing_df = load_existing_results()
    new_rows = []

    for index, experiment in enumerate(experiments, start=1):
        print("=" * 100)
        print(
            f"[{index}/{len(experiments)}] {experiment['id']} - {experiment['name']}"
            f" - seed={seed} - {epochs} epoch - {hp_source_for(experiment)}"
        )
        print(experiment["description"])

        if SKIP_COMPLETED and has_successful_run(existing_df, experiment, seed, epochs):
            print("이미 성공한 결과가 있어 건너뜁니다.")
            continue

        try:
            row = train_one_experiment(experiment, seed=seed, epochs=epochs)
        except Exception as error:
            row = {
                "status": "FAILED",
                "id": experiment["id"],
                "name": experiment["name"],
                "family": experiment["family"],
                "seed": seed,
                "requested_epochs": epochs,
                "hp_mode": experiment.get("hp_mode", "tuned"),
                "hp_source": hp_source_for(experiment),
                "error": repr(error),
            }

        new_rows.append(row)

        current_existing = load_existing_results()
        combined = pd.concat(
            [current_existing, pd.DataFrame([row])],
            ignore_index=True,
        )

        # 같은 id+seed가 여러 번 존재하면 가장 최근 행을 유지합니다.
        dedup_keys = [
            key
            for key in ["id", "seed", "requested_epochs", "hp_source"]
            if key in combined.columns
        ]

        if dedup_keys:
            combined = combined.drop_duplicates(
                subset=dedup_keys,
                keep="last",
            )

        combined.to_csv(
            RESULTS_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        display(pd.DataFrame([row]))

    return load_existing_results()

## 25. 증강 재실험 실행

In [ ]:
if RUN_EXPERIMENTS:
    retune_results_df = run_experiment_list(
        EXPERIMENTS,
        seed=SEED,
        epochs=EPOCHS,
    )

else:
    print("RUN_EXPERIMENTS=False")
    retune_results_df = load_existing_results()

display(retune_results_df)


## 26. 최종 비교

In [ ]:
set_korean_font()

results_df = load_existing_results()

compare_df = results_df[
    results_df["status"].eq("OK")
    & results_df["requested_epochs"].eq(EPOCHS)
].drop_duplicates(subset=["id"], keep="last")

order = [key for key in CONTROL_IDS + AUG_SHORTLIST_IDS if key in set(compare_df["id"])]
compare_df = compare_df.set_index("id").loc[order].reset_index()

display(
    compare_df[
        ["id", "name", "hp_source", "precision", "recall", "f1", "mAP50", "mAP50_95"]
    ].round(4)
)

RETUNE_COMPARE_CSV = SUMMARY_DIR / "retune_compare.csv"
compare_df.to_csv(RETUNE_COMPARE_CSV, index=False, encoding="utf-8-sig")
print("Saved:", RETUNE_COMPARE_CSV)


# ------------------------------------------------------------
# 02번(옛 하이퍼파라미터) 대비 개선폭
# ------------------------------------------------------------
old_map = (
    nb02_df[nb02_df["status"].eq("OK") & nb02_df["requested_epochs"].eq(EPOCHS)]
    .drop_duplicates(subset=["id"], keep="last")
    .set_index("id")["mAP50_95"]
)

compare_df["02번_mAP50_95"] = compare_df["id"].map(old_map)
compare_df["개선폭"] = compare_df["mAP50_95"] - compare_df["02번_mAP50_95"]

display(compare_df[["id", "name", "02번_mAP50_95", "mAP50_95", "개선폭"]].round(4))

plottable = compare_df[compare_df["개선폭"].notna()]

if len(plottable):
    plt.figure(figsize=(10, max(5, len(plottable) * 0.45)))

    bars = plt.barh(
        plottable["id"] + "  " + plottable["name"],
        plottable["개선폭"],
        color=["#2e7d32" if v > 0 else "#c62828" for v in plottable["개선폭"]],
    )

    for bar, value in zip(bars, plottable["개선폭"]):
        plt.text(value, bar.get_y() + bar.get_height() / 2, f" {value:+.4f}", va="center")

    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("mAP50-95 변화량 (05번 - 02번)")
    plt.title(f"17개 카테고리 재튜닝 효과 ({EPOCHS} epoch, 같은 증강끼리 비교)")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    figure_path = FINAL_DIR / "retune_gain.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)


## 26-1. 증강 조합 상위 2개 저장

06번이 교차 실험에 쓸 증강 두 개를 고릅니다.
대조군(B00~B03)은 제외하고, 이번 재실험 결과 기준으로 순위를 매깁니다.

**주의해서 볼 것**: 1위와 2위의 차이가 아주 작으면 둘 다 사실상 같은 성능입니다.
02번에서도 multi-seed 상위 3개가 0.7992 / 0.7976 / 0.7956(표준편차 0.02 이상)이었고,
그때 1위였던 `C11`이 15 epoch에서는 최하위가 됐습니다.
차이가 0.005 미만이면 "증강 간 차이 없음"으로 읽는 편이 안전합니다.

In [ ]:
TOP2_AUG_JSON = SUMMARY_DIR / "top2_augmentations.json"

augmentation_rank = (
    compare_df[~compare_df["id"].isin(CONTROL_IDS)]
    .sort_values("mAP50_95", ascending=False)
    .reset_index(drop=True)
)

display(augmentation_rank[["id", "name", "mAP50_95", "recall", "f1"]].round(4))

top2_aug_records = []

for rank, (_, row) in enumerate(augmentation_rank.head(2).iterrows(), start=1):
    top2_aug_records.append({
        "rank": rank,
        "label": f"aug{rank}",
        "id": str(row["id"]),
        "name": str(row["name"]),
        "mAP50_95": float(row["mAP50_95"]),
    })

gap = None

if len(top2_aug_records) == 2:
    gap = top2_aug_records[0]["mAP50_95"] - top2_aug_records[1]["mAP50_95"]

top2_aug_payload = {
    "dataset": DATASET_FINGERPRINT,
    "epochs": EPOCHS,
    "rank_gap_1_to_2": gap,
    "augmentations": top2_aug_records,
}

with open(TOP2_AUG_JSON, "w", encoding="utf-8") as file:
    json.dump(top2_aug_payload, file, ensure_ascii=False, indent=2)

print()
for record in top2_aug_records:
    print(f"[{record['label']}] {record['id']} {record['name']}  "
          f"mAP50-95 {record['mAP50_95']:.4f}")

if gap is not None:
    print()
    print(f"1위와 2위 차이: {gap:.4f}")

    if gap < 0.005:
        print("-> seed 노이즈 수준입니다. 두 조합의 우열을 단정하지 마세요.")

print()
print("Saved:", TOP2_AUG_JSON)


## 27. 다음 단계

이 노트북이 06번에 넘기는 파일은 두 개입니다.

```text
report/optuna/top2_hyperparameters.json   하이퍼파라미터 hp1 / hp2
report/summary/top2_augmentations.json    증강 조합 aug1 / aug2
```

06번은 이 둘을 교차해 4가지(hp1xaug1, hp1xaug2, hp2xaug1, hp2xaug2)를 만들고,
여기에 baseline 4가지(B00~B03)를 더해 총 8가지를 15 epoch으로 실행합니다.

## 결과를 읽을 때

- `B03 - B00`이 이번에도 0 근처이거나 음수라면, 이 데이터셋에서도
  하이퍼파라미터 튜닝의 여지가 크지 않다는 뜻입니다.
- 차이가 ±0.005~0.02 수준이면 seed 하나 차이로도 나올 수 있는 범위입니다.
- 지금까지 86개 / 17개 clean / 17개 파손포함 세 번 모두
  **B01(YOLO 기본 증강 + auto)** 이 가장 좋았습니다. 이번에도 그렇다면
  "이 데이터에서는 기본 설정이 이미 충분히 좋다"가 정직한 결론입니다.
